# Task_3_Build_Execute
## Loan Eligibility Model


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ─── Load Dataset ──────────────────────────────────────────────────────────
# Run Task 2 script first to generate loan_applicants.csv
# For standalone use, we recreate it here
np.random.seed(42)
n = 500
data = {
    "age": np.random.randint(21, 60, n),
    "gender": np.random.choice([0, 1], n),  # 0=Female, 1=Male
    "income_monthly": np.random.randint(5000, 80000, n),
    "employment_type": np.random.choice([0, 1, 2, 3], n, p=[0.4, 0.3, 0.2, 0.1]),  # Salaried/Self/Business/Unemployed
    "loan_amount": np.random.randint(10000, 500000, n),
    "loan_term_months": np.random.choice([12, 24, 36, 48, 60], n),
    "num_dependents": np.random.randint(0, 6, n),
    "credit_history_score": np.random.randint(300, 900, n),
    "existing_loans": np.random.randint(0, 5, n),
}
df = pd.DataFrame(data)
df["eligible"] = (
    (df["credit_history_score"] >= 600) &
    (df["income_monthly"] >= 15000) &
    (df["employment_type"] != 3) &
    (df["loan_amount"] <= df["income_monthly"] * 12)
).astype(int)

# ─── Feature Engineering ───────────────────────────────────────────────────
df["loan_to_income_ratio"] = df["loan_amount"] / (df["income_monthly"] * df["loan_term_months"])

X = df.drop("eligible", axis=1)
y = df["eligible"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ─── Model 1: Logistic Regression ─────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_preds = lr_model.predict(X_test_scaled)
lr_acc = accuracy_score(y_test, lr_preds)
print(f"Logistic Regression Accuracy: {lr_acc:.2%}")

# ─── Model 2: Random Forest ────────────────────────────────────────────────
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Random Forest Accuracy:        {rf_acc:.2%}")

# ─── Best Model Report ────────────────────────────────────────────────────
best_model = rf_model if rf_acc > lr_acc else lr_model
print("\n=== Classification Report (Random Forest) ===")
print(classification_report(y_test, rf_preds, target_names=["Rejected", "Approved"]))

# Feature importance
feature_importance = pd.Series(rf_model.feature_importances_, index=X.columns)
print("\n=== Top Feature Importances ===")
print(feature_importance.sort_values(ascending=False).head(5))